# Feature Engineering for Drug Intelligence

## Objective

The objective of this notebook is to create meaningful features from both structured and unstructured data to support predictive modeling and decision-making.

## Key Tasks

* Combine NLP outputs with structured features
* Create demand proxy variable
* Generate drug-level features
* Prepare final dataset for modeling

## Importance

This step bridges raw insights and machine learning by transforming data into actionable features.

## Output

A final dataset ready for:

* effectiveness prediction
* demand forecasting
* decision engine


In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/nlp_processed_reviews.csv")
drug_nlp_df = pd.read_csv("../data/drug_nlp_summary.csv")

# Demand Feature Creation

Since real-world sales data is not available, we create a proxy demand variable using:

* review count
* sentiment score
* usefulness

This simulates real-world business demand.


In [22]:
drug_nlp_df['demand'] = (
    drug_nlp_df['review_count'] * 5 +
    drug_nlp_df['avg_sentiment'] * 100 +
    drug_nlp_df['avg_usefulness'] * 2 +
    np.random.randint(0, 50, size=len(drug_nlp_df))
)

In [23]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
drug_nlp_df['demand_scaled'] = scaler.fit_transform(drug_nlp_df[['demand']])

# Effectiveness Score

We use average rating as a proxy for drug effectiveness.

This will be used as a target variable for prediction models.


In [24]:
drug_nlp_df['effectiveness_score'] = drug_nlp_df['avg_rating']

In [25]:
drug_nlp_df['sentiment_per_review'] = (
    drug_nlp_df['avg_sentiment'] / (drug_nlp_df['review_count'] + 1)
)

drug_nlp_df['engagement_score'] = (
    drug_nlp_df['avg_usefulness'] * drug_nlp_df['review_count']
)

In [ ]:
drug_nlp_df['popularity'] = pd.qcut(
    drug_nlp_df['review_count'], 
    q=3, 
    labels=['low', 'medium', 'high']
)

In [29]:
# strong proxy features for effectiveness

drug_nlp_df['sentiment_rating_proxy'] = (
    drug_nlp_df['avg_sentiment'] * drug_nlp_df['review_count']
)

drug_nlp_df['sentiment_usefulness'] = (
    drug_nlp_df['avg_sentiment'] * drug_nlp_df['avg_usefulness']
)

drug_nlp_df['rating_variation_proxy'] = (
    drug_nlp_df['review_count'] / (drug_nlp_df['avg_usefulness'] + 1)
)

In [30]:
final_df = drug_nlp_df[[
    'drugName',
    'avg_sentiment',
    'avg_rating',
    'review_count',
    'avg_usefulness',
    'demand',
    'demand_scaled',
    'effectiveness_score',
    'sentiment_per_review',
    'engagement_score',
    'popularity',
    'top_side_effects',

    'sentiment_rating_proxy',
    'sentiment_usefulness',
    'rating_variation_proxy'
]]

In [31]:
final_df.to_csv("../data/Final_data/final_drug_dataset.csv", index=False)